# ProofHound — train the 7B Harness model (QLoRA) on Kaggle

Kaggle twin of `training/train/colab_train_harness.ipynb`: same `train_local.py`, same 7B settings, but data comes from the private dataset `proofhound-harness-data` and the adapter is written to `/kaggle/working/harness-adapter-7b` (the notebook output). Pinned to one GPU so the result shows the 7B fits a single 16 GB card.

Pushed and run from the laptop with `kaggle kernels push -p training/train/kaggle`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # 14B needs both T4s via device_map="auto"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
assert torch.cuda.is_available(), "No GPU: this notebook must run with a GPU accelerator"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB",
      "| compute capability", torch.cuda.get_device_capability(0), "| torch", torch.__version__)
# Fails in seconds if this PyTorch build has no kernels for the card (e.g. P100 on a newer build).
print("cuda op ok:", (torch.ones(4, device="cuda") * 2).sum().item())


In [ ]:
MODEL          = "Qwen/Qwen2.5-Coder-14B-Instruct"
MAX_LENGTH     = 2048
LORA_RANK      = 16
TARGET_MODULES = "q_proj k_proj v_proj o_proj"
EPOCHS         = 3
SAVE_STEPS     = 25
OUT            = "/kaggle/working/harness-adapter-14b-v6"
REPO           = "/tmp/proofhound"


In [ ]:
if not os.path.isdir(REPO):
    !git clone -q --depth 1 https://github.com/moabdelmoati/proofhound.git {REPO}
%cd {REPO}
!git log -1 --oneline

In [ ]:
!pip -q install -e . "transformers==5.20.0" "peft==0.21.2" bitsandbytes accelerate

In [ ]:
import bitsandbytes as bnb
layer = bnb.nn.Linear4bit(64, 64, compute_dtype=torch.float16, quant_type="nf4").cuda()
print("bitsandbytes 4-bit ok:", tuple(layer(torch.randn(2, 64, device="cuda", dtype=torch.float16)).shape))

In [ ]:
import glob, shutil
os.makedirs("training/data", exist_ok=True)
for name in ("harness_train.jsonl", "harness_attempts.jsonl"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert found, f"{name} not found under /kaggle/input"
    shutil.copy(found[0], "training/data/")
DATA = "training/data/harness_train.jsonl"
print(os.listdir("training/data"))

In [ ]:
!python training/train/train_local.py --model {MODEL} --max-length {MAX_LENGTH} --lora-rank {LORA_RANK} --target-modules {TARGET_MODULES} --epochs {EPOCHS} --save-steps {SAVE_STEPS} --data {DATA} --out "{OUT}"

In [ ]:
assert os.path.exists(f"{OUT}/adapter_config.json"), "training did not save an adapter"
for checkpoint in glob.glob(f"{OUT}/checkpoint-*"):
    shutil.rmtree(checkpoint)
for f in sorted(os.listdir(OUT)):
    print(f"{f:40s} {os.path.getsize(os.path.join(OUT, f)) / 2**20:8.1f} MB")

In [ ]:
!python training/serve/generate_test.py --adapter "{OUT}" --device cuda --load-4bit --prompts training/data/harness_attempts.jsonl